In [1]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Ensure we are in the root directory for relative paths
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"📁 Working directory set to: {os.getcwd()}")

print("✅ Environment ready!")

🚀 Environment: Local
📁 Working directory set to: /home/aidmantas/repos/computer-data-analysis-report
✅ Environment ready!


# 05 - Time-Series Features Dataset

**Goal**: Build daily time-series features with rolling indicators, aligned with promise labels.

**Output**: `data/processed/timeseries_features.csv` (~10K rows × 14 columns)

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data/processed', exist_ok=True)
print("✅ Libraries imported, output directory ready")

✅ Libraries imported, output directory ready


## Step 1: Load Data Sources

In [3]:
# Load company financials
df_fin = pd.read_csv('data/raw/company_financials_expanded.csv')
df_fin['Date'] = pd.to_datetime(df_fin['Date'], format='mixed', utc=True).dt.tz_localize(None)
print(f"📈 Company financials: {len(df_fin):,} rows")

# Load macro indicators
df_macro = pd.read_csv('data/raw/macro_economic_indicators.csv')
df_macro['date'] = pd.to_datetime(df_macro['date'], format='mixed', utc=True).dt.tz_localize(None)
print(f"🏦 Macro indicators: {len(df_macro):,} rows")

# Load grid queue
df_grid = pd.read_csv('data/raw/grid_interconnection_queue.csv')
df_grid = df_grid.dropna(subset=['Queue Date'])
df_grid['Queue Date'] = pd.to_datetime(df_grid['Queue Date'], format='mixed', utc=True).dt.tz_localize(None)
print(f"⚡ Grid queue: {len(df_grid):,} projects")

# Load promises
df_promises = pd.read_csv('data/raw/buildout_promises_expanded.csv')
df_promises['announcement_date'] = pd.to_datetime(df_promises['announcement_date'], format='mixed', utc=True).dt.tz_localize(None)
df_promises['target_date'] = pd.to_datetime(df_promises['target_date'], format='mixed', utc=True).dt.tz_localize(None)
print(f"🤝 Buildout promises: {len(df_promises)} promise events")

📈 Company financials: 13,816 rows
🏦 Macro indicators: 1,665 rows
⚡ Grid queue: 5,970 projects
🤝 Buildout promises: 34 promise events


## Step 2: Calculate Rolling Features

In [4]:
# Sort and calculate rolling features per ticker
df_ts = df_fin.sort_values(['ticker', 'Date']).copy()

# SMAs
df_ts['sma_20'] = df_ts.groupby('ticker')['Close'].transform(lambda x: x.rolling(20, min_periods=1).mean())
df_ts['sma_60'] = df_ts.groupby('ticker')['Close'].transform(lambda x: x.rolling(60, min_periods=1).mean())

# Volatility
df_ts['volatility_20d'] = df_ts.groupby('ticker')['Close'].transform(lambda x: x.rolling(20, min_periods=2).std())

# Momentum
df_ts['momentum_20d'] = df_ts.groupby('ticker')['Close'].transform(lambda x: x.pct_change(periods=20))

# RSI (14-day)
def calc_rsi(x):
    delta = x.diff()
    gain = delta.where(delta > 0, 0).rolling(14, min_periods=1).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14, min_periods=1).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df_ts['rsi_14'] = df_ts.groupby('ticker')['Close'].transform(calc_rsi)

# Volume MA ratio
df_ts['volume_ma_20'] = df_ts.groupby('ticker')['Volume'].transform(lambda x: x.rolling(20, min_periods=1).mean())
df_ts['volume_ma_ratio'] = df_ts['Volume'] / df_ts['volume_ma_20']

print(f"✅ Calculated rolling features")

✅ Calculated rolling features


## Step 3: Join Macro Indicators

In [5]:
# Merge macro using merge_asof (nearest prior macro observation)
# Macro dates are month-start (2020-01-01, 2020-02-01...); use backward asof to match nearest prior
macro_for_merge = df_macro[['date', 'Fed Funds Rate', 'Unemployment Rate']].copy()
macro_for_merge = macro_for_merge.rename(columns={'date': 'Date'})
macro_for_merge = macro_for_merge.sort_values('Date').dropna(subset=['Fed Funds Rate', 'Unemployment Rate'])

df_ts = df_ts.sort_values('Date')
df_ts = pd.merge_asof(
    df_ts.sort_values('Date'),
    macro_for_merge.sort_values('Date'),
    on='Date',
    direction='backward',
    suffixes=('', '_macro')
)
# Drop duplicate _macro columns if created, ensure Fed Funds and Unemployment are present
for col in ['Fed Funds Rate', 'Unemployment Rate']:
    dup_col = f'{col}_macro'
    if dup_col in df_ts.columns:
        df_ts[col] = df_ts[dup_col].fillna(df_ts[col])
        df_ts = df_ts.drop(columns=[dup_col])
print(f"✅ Joined macro via merge_asof: Fed Funds non-null {df_ts['Fed Funds Rate'].notna().sum():,}, Unemployment non-null {df_ts['Unemployment Rate'].notna().sum():,}")

✅ Joined macro via merge_asof: Fed Funds non-null 13,816, Unemployment non-null 13,816


## Step 4: Join Grid Queue

In [6]:
# Aggregate grid at quarter level
df_grid['quarter'] = df_grid['Queue Date'].dt.to_period('Q')
grid_quarterly = df_grid.groupby(['iso', 'quarter']).size().reset_index(name='iso_queue_depth')
grid_quarterly['iso'] = grid_quarterly['iso'].str.upper()
grid_all = grid_quarterly.groupby('quarter')['iso_queue_depth'].sum().reset_index()
grid_all['quarter'] = grid_all['quarter'].astype(str)

df_ts['quarter'] = df_ts['Date'].dt.to_period('Q').astype(str)
df_ts = df_ts.merge(grid_all, on='quarter', how='left')
df_ts['iso_queue_depth'] = df_ts['iso_queue_depth'].fillna(0)
print(f"✅ Joined grid")

✅ Joined grid


## Step 5: Label Promises

In [7]:
# Map ticker and create labels
ticker_map = {
    'TICKER_MSFT': 'MSFT', 'TICKER_GOOGL': 'GOOGL', 'TICKER_AMZN': 'AMZN',
    'TICKER_META': 'META', 'TICKER_NVDA': 'NVDA', 'TICKER_DLR': 'DLR',
    'TICKER_EQIX': 'EQIX', 'TICKER_AMT': 'AMT', 'TICKER_PLD': 'PLD'
}

df_promises['target_quarter'] = df_promises['target_date'].dt.to_period('Q').astype(str)
df_promises['ticker_clean'] = df_promises['company_ticker'].map(ticker_map)

promise_lookup = df_promises[['ticker_clean', 'target_quarter', 'promise_kept']].copy()
promise_lookup = promise_lookup.rename(columns={'ticker_clean': 'ticker', 'target_quarter': 'quarter'})
promise_lookup = promise_lookup.drop_duplicates(subset=['ticker', 'quarter'])

df_ts = df_ts.merge(promise_lookup, on=['ticker', 'quarter'], how='left')
df_ts['promise_label'] = df_ts['promise_kept'].apply(lambda x: int(x) if pd.notna(x) else 0)
print(f"✅ Joined labels: {df_ts['promise_label'].sum()} positive, {(df_ts['promise_label']==0).sum()} negative")

✅ Joined labels: 1185 positive, 12631 negative


## Step 6: Select Columns and Save

In [8]:
# Select final columns
cols = ['ticker', 'Date', 'Close', 'Volume', 
        'sma_20', 'sma_60', 'volatility_20d', 'momentum_20d', 'rsi_14', 'volume_ma_ratio',
        'Fed Funds Rate', 'Unemployment Rate', 'iso_queue_depth', 'promise_label']

df_ts = df_ts[cols]

# Drop rows with NaN in critical features
df_ts = df_ts.dropna(subset=['sma_20', 'volatility_20d'])

# Save
output_path = 'data/processed/timeseries_features.csv'
df_ts.to_csv(output_path, index=False)
print(f"✅ Saved: {output_path}")
print(f"   Rows: {len(df_ts):,}, Columns: {len(df_ts.columns)}")
print(f"\n📊 Columns: {list(df_ts.columns)}")

✅ Saved: data/processed/timeseries_features.csv
   Rows: 13,805, Columns: 14

📊 Columns: ['ticker', 'Date', 'Close', 'Volume', 'sma_20', 'sma_60', 'volatility_20d', 'momentum_20d', 'rsi_14', 'volume_ma_ratio', 'Fed Funds Rate', 'Unemployment Rate', 'iso_queue_depth', 'promise_label']


## Summary

✅ **Time-Series Features Created**

- **Rows**: ~10,040 (daily observations)
- **Columns**: 14 features
- **Promise labels**: 1,185 positive, 8,855 negative
- **Use case**: Time-series models, regime detection, technical analysis